# SpineView — Cervical Landmark Detection Model

Train a lightweight keypoint detection model on the CSXA dataset for
accurate cervical lateral X-ray landmark detection.

**Output:** ONNX model (~15MB) that runs in-browser via ONNX Runtime Web.

## Setup
1. Download CSXA V3.0 from https://www.scidb.cn/en/detail?dataSetId=8e3b3d5e60a348ba961e19d48b881c90
2. Upload the zip to Google Drive
3. Run this notebook on Google Colab (GPU runtime)

In [ ]:
# ─── Install Dependencies ────────────────────────────────
!pip install -q torch torchvision timm albumentations opencv-python-headless onnx onnxruntime pillow tqdm scikit-learn

In [ ]:
# ─── Mount Google Drive ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DATASET_ZIP = '/content/drive/MyDrive/dataset/datasets-JSON.zip'
WORK_DIR = '/content/spine_training'
DATA_DIR = f'{WORK_DIR}/datasets-JSON'
OUTPUT_DIR = f'{WORK_DIR}/output'
IMAGES_DIR = f'{WORK_DIR}/images'  # We'll extract embedded images here

In [ ]:
# ─── Extract Dataset ─────────────────────────────────────
import os, zipfile, json, base64
from pathlib import Path
from tqdm import tqdm

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)

# Extract zip if needed
if not os.path.exists(DATA_DIR):
    print('Extracting dataset zip...')
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall(WORK_DIR)
    print(f'Extracted to {DATA_DIR}')

json_files = sorted(Path(DATA_DIR).glob('*.json'))
print(f'Found {len(json_files)} JSON annotation files')

# Extract embedded images from JSON (CSXA stores base64 imageData inside each JSON)
existing_images = set(os.listdir(IMAGES_DIR))
extracted = 0
for jf in tqdm(json_files, desc='Extracting images'):
    img_out = f'{IMAGES_DIR}/{jf.stem}.png'
    if f'{jf.stem}.png' in existing_images:
        continue
    with open(jf) as f:
        ann = json.load(f)
    img_data = ann.get('imageData')
    if img_data:
        img_bytes = base64.b64decode(img_data)
        with open(img_out, 'wb') as out:
            out.write(img_bytes)
        extracted += 1

print(f'Extracted {extracted} new images to {IMAGES_DIR}')
print(f'Total images: {len(os.listdir(IMAGES_DIR))}')

In [ ]:
# ─── Verify Data ─────────────────────────────────────────
import json

# Check a sample annotation
json_files = sorted(Path(DATA_DIR).glob('*.json'))
with open(json_files[0]) as f:
    sample = json.load(f)

print(f'Sample file: {json_files[0].name}')
print(f'Image size: {sample.get("imageWidth")}x{sample.get("imageHeight")}')
print(f'Keypoints found:')
for shape in sample.get('shapes', []):
    label = shape['label']
    pt = shape['points'][0]
    print(f'  {label}: ({pt[0]:.1f}, {pt[1]:.1f})')

# Verify matching image exists
img_check = f'{IMAGES_DIR}/{json_files[0].stem}.png'
print(f'\nImage file exists: {os.path.exists(img_check)}')

In [ ]:
# ─── Data Loading & Mapping ──────────────────────────────
# CSXA uses labelme format. Each JSON has a "shapes" array where
# each shape has a "label" (keypoint name) and "points" [[x, y]].
#
# CSXA 23 keypoints:
#   C2 centroid, C2 bottom left, C2 bottom right
#   C3-C7: top left, top right, bottom left, bottom right
#
# In lateral X-rays: "left" = posterior, "right" = anterior
# "top" = superior, "bottom" = inferior

import json
import numpy as np
from pathlib import Path

# ── SpineView landmark order (must match posture-pro constants.ts) ──
SPINEVIEW_LANDMARKS = [
    'C1_post',       # C1 posterior tubercle (NOT in CSXA — will be invisible)
    'C2_sup_post',   # C2 superior-posterior (NOT in CSXA — will be invisible)
    'C2_inf_post',   # C2 inferior-posterior = "C2 bottom left"
    'C3_sup_post',   # C3 superior-posterior = "C3 top left"
    'C3_inf_post',   # C3 inferior-posterior = "C3 bottom left"
    'C4_sup_post',   # C4 superior-posterior = "C4 top left"
    'C4_inf_post',   # C4 inferior-posterior = "C4 bottom left"
    'C5_sup_post',   # C5 superior-posterior = "C5 top left"
    'C5_inf_post',   # C5 inferior-posterior = "C5 bottom left"
    'C6_sup_post',   # C6 superior-posterior = "C6 top left"
    'C6_inf_post',   # C6 inferior-posterior = "C6 bottom left"
    'C7_sup_post',   # C7 superior-posterior = "C7 top left"
    'C7_inf_post',   # C7 inferior-posterior = "C7 bottom left"
    'T1_sup_post',   # T1 superior-posterior (NOT in CSXA — will be invisible)
    'C2_sup_ant',    # C2 superior-anterior (NOT in CSXA — will be invisible)
    'C2_inf_ant',    # C2 inferior-anterior = "C2 bottom right"
    'C7_inf_ant',    # C7 inferior-anterior = "C7 bottom right"
]
NUM_KEYPOINTS = len(SPINEVIEW_LANDMARKS)
print(f'SpineView has {NUM_KEYPOINTS} keypoints')

# ── CSXA labelme label → SpineView landmark ID ──
# In CSXA lateral views: left=posterior, right=anterior, top=superior, bottom=inferior
CSXA_TO_SPINEVIEW = {
    # C2 (only bottom corners + centroid in CSXA)
    'C2 bottom left':  'C2_inf_post',
    'C2 bottom right': 'C2_inf_ant',
    # C3
    'C3 top left':     'C3_sup_post',
    'C3 bottom left':  'C3_inf_post',
    # C4
    'C4 top left':     'C4_sup_post',
    'C4 bottom left':  'C4_inf_post',
    # C5
    'C5 top left':     'C5_sup_post',
    'C5 bottom left':  'C5_inf_post',
    # C6
    'C6 top left':     'C6_sup_post',
    'C6 bottom left':  'C6_inf_post',
    # C7
    'C7 top left':     'C7_sup_post',
    'C7 bottom left':  'C7_inf_post',
    'C7 bottom right': 'C7_inf_ant',
}

# Landmarks NOT in CSXA that the model will learn are "invisible":
# C1_post, C2_sup_post, T1_sup_post, C2_sup_ant
# These will have visibility=0 during training.

print(f'Mapping {len(CSXA_TO_SPINEVIEW)} CSXA keypoints → SpineView landmarks')
print(f'{NUM_KEYPOINTS - len(CSXA_TO_SPINEVIEW)} landmarks will be marked invisible (not in CSXA)')

In [ ]:
# ─── Dataset Class ───────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

INPUT_SIZE = 384  # Model input resolution
HEATMAP_SIZE = 96  # Heatmap output resolution (INPUT_SIZE / 4)
SIGMA = 2.5       # Gaussian sigma for heatmap generation


def generate_heatmap(keypoints, heatmap_size, sigma):
    """Generate NUM_KEYPOINTS heatmaps from normalised keypoint coords."""
    heatmaps = np.zeros((NUM_KEYPOINTS, heatmap_size, heatmap_size), dtype=np.float32)
    for i, (x, y, vis) in enumerate(keypoints):
        if vis < 0.5:  # Not visible
            continue
        cx = int(x * heatmap_size)
        cy = int(y * heatmap_size)
        if cx < 0 or cx >= heatmap_size or cy < 0 or cy >= heatmap_size:
            continue
        # 2D Gaussian
        xx, yy = np.meshgrid(np.arange(heatmap_size), np.arange(heatmap_size))
        heatmaps[i] = np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * sigma**2))
    return heatmaps


class CSXADataset(Dataset):
    def __init__(self, image_paths, annotations, transform=None):
        self.image_paths = image_paths
        self.annotations = annotations  # List of (N, 3) arrays: [x_norm, y_norm, visibility]
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(str(self.image_paths[idx]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        kpts = self.annotations[idx].copy()  # (NUM_KEYPOINTS, 3)

        h_orig, w_orig = img.shape[:2]

        if self.transform:
            # Convert normalised coords to pixel coords for albumentations
            kpts_px = []
            for x, y, v in kpts:
                kpts_px.append((x * w_orig, y * h_orig, 0, 0))  # x, y, angle, scale

            transformed = self.transform(
                image=img,
                keypoints=kpts_px,
                keypoint_params=A.KeypointParams(format='xy', remove_invisible=False),
            )
            img = transformed['image']
            kpts_px_out = transformed['keypoints']

            # Convert back to normalised coords
            for i, (kx, ky, *_) in enumerate(kpts_px_out):
                kpts[i, 0] = kx / INPUT_SIZE
                kpts[i, 1] = ky / INPUT_SIZE
        else:
            img = cv2.resize(img, (INPUT_SIZE, INPUT_SIZE))
            img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        heatmaps = generate_heatmap(kpts, HEATMAP_SIZE, SIGMA)

        return img, torch.from_numpy(heatmaps), torch.from_numpy(kpts)


# ── Augmentation pipeline ──
train_transform = A.Compose(
    [
        A.Resize(INPUT_SIZE, INPUT_SIZE),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussNoise(var_limit=(5, 25), p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ],
    keypoint_params=A.KeypointParams(format='xy', remove_invisible=False),
)

val_transform = A.Compose(
    [
        A.Resize(INPUT_SIZE, INPUT_SIZE),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ],
    keypoint_params=A.KeypointParams(format='xy', remove_invisible=False),
)

print(f'Dataset class ready. Input: {INPUT_SIZE}x{INPUT_SIZE}, Output heatmap: {HEATMAP_SIZE}x{HEATMAP_SIZE}')

In [ ]:
# ─── Load & Split Data ───────────────────────────────────
from sklearn.model_selection import train_test_split

def load_csxa_data(data_dir, images_dir, mapping):
    """Load CSXA labelme annotations and map to SpineView format."""
    image_paths = []
    annotations = []

    json_files = sorted(Path(data_dir).glob('*.json'))
    print(f'Processing {len(json_files)} annotation files...')

    skipped = 0
    format_errors = 0
    for jf in json_files:
        with open(jf) as f:
            ann = json.load(f)

        # Image is in the extracted images directory
        img_path = Path(images_dir) / f'{jf.stem}.png'
        if not img_path.exists():
            skipped += 1
            continue

        # Get image dimensions from JSON
        w = ann.get('imageWidth', 0)
        h = ann.get('imageHeight', 0)
        if w == 0 or h == 0:
            from PIL import Image as PILImage
            pil_img = PILImage.open(img_path)
            w, h = pil_img.size

        # Parse labelme shapes into a lookup dict: label → (x, y)
        shapes = ann.get('shapes', [])
        if not shapes:
            format_errors += 1
            continue

        label_to_point = {}
        for shape in shapes:
            label = shape.get('label', '')
            points = shape.get('points', [])
            if label and points and len(points) > 0:
                pt = points[0]
                if isinstance(pt, (list, tuple)) and len(pt) >= 2:
                    label_to_point[label] = (pt[0], pt[1])

        # Map CSXA keypoints to SpineView landmarks
        kpts = np.zeros((NUM_KEYPOINTS, 3), dtype=np.float32)
        mapped = 0
        for csxa_label, sv_id in mapping.items():
            if csxa_label in label_to_point:
                sv_idx = SPINEVIEW_LANDMARKS.index(sv_id)
                px, py = label_to_point[csxa_label]
                kpts[sv_idx] = [px / w, py / h, 1.0]
                mapped += 1

        if mapped >= 8:
            image_paths.append(str(img_path))
            annotations.append(kpts)

    print(f'Loaded {len(image_paths)} samples')
    print(f'  Skipped {skipped} (no image), {format_errors} (no shapes)')
    return image_paths, annotations


# Load data
image_paths, annotations = load_csxa_data(DATA_DIR, IMAGES_DIR, CSXA_TO_SPINEVIEW)

# Quick sanity check
if len(image_paths) > 0:
    sample_kpts = annotations[0]
    visible = sum(1 for x, y, v in sample_kpts if v > 0.5)
    print(f'\nSample: {visible}/{NUM_KEYPOINTS} landmarks visible')
    for i, (x, y, v) in enumerate(sample_kpts):
        if v > 0.5:
            print(f'  {SPINEVIEW_LANDMARKS[i]:15s}: ({x:.3f}, {y:.3f})')
else:
    print('\n⚠️ No samples loaded! Check the mapping and file paths.')

# 85/15 train/val split
train_paths, val_paths, train_anns, val_anns = train_test_split(
    image_paths, annotations, test_size=0.15, random_state=42
)

train_ds = CSXADataset(train_paths, train_anns, transform=train_transform)
val_ds = CSXADataset(val_paths, val_anns, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f'\nTrain: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
# ─── Model: Lightweight HRNet-W18 ────────────────────────
import timm
import torch.nn as nn


class SpineLandmarkNet(nn.Module):
    """HRNet-W18 backbone with heatmap regression head."""

    def __init__(self, num_keypoints=NUM_KEYPOINTS, pretrained=True):
        super().__init__()
        # HRNet-W18 — lightweight, high-resolution representations
        self.backbone = timm.create_model(
            'hrnet_w18_small_v2',
            pretrained=pretrained,
            features_only=True,
        )

        # Get the output channels of the highest-resolution feature map
        with torch.no_grad():
            dummy = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE)
            feats = self.backbone(dummy)
            feat_channels = feats[0].shape[1]
            feat_size = feats[0].shape[2]
            print(f'Backbone output: {feat_channels} channels, {feat_size}x{feat_size} spatial')

        # Heatmap head
        self.head = nn.Sequential(
            nn.Conv2d(feat_channels, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, num_keypoints, 1),
        )

        # Upsample to match target heatmap size if needed
        self.upsample = nn.Upsample(size=(HEATMAP_SIZE, HEATMAP_SIZE), mode='bilinear', align_corners=False)

    def forward(self, x):
        feats = self.backbone(x)
        heatmaps = self.head(feats[0])  # Use highest-resolution feature map
        heatmaps = self.upsample(heatmaps)
        return heatmaps


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = SpineLandmarkNet(NUM_KEYPOINTS).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params / 1e6:.1f}M')

In [ ]:
# ─── Training Loop ───────────────────────────────────────
from tqdm import tqdm
import shutil

EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-4
CHECKPOINT_EVERY = 5  # Save to Google Drive every N epochs
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/SpineView_Model'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.MSELoss()

# Resume from checkpoint if one exists on Drive
resume_path = f'{DRIVE_CHECKPOINT_DIR}/checkpoint_latest.pth'
start_epoch = 0
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'val_pck': []}

if os.path.exists(resume_path):
    print(f'Resuming from checkpoint...')
    ckpt = torch.load(resume_path, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt['best_val_loss']
    history = ckpt['history']
    print(f'  Resumed at epoch {start_epoch}, best_val_loss={best_val_loss:.6f}')
else:
    print(f'Starting fresh training for {EPOCHS} epochs')


def pck_metric(pred_heatmaps, gt_keypoints, threshold=0.05):
    """Percentage of Correct Keypoints within threshold of normalised distance."""
    batch_size = pred_heatmaps.shape[0]
    correct = 0
    total = 0

    for b in range(batch_size):
        for k in range(NUM_KEYPOINTS):
            if gt_keypoints[b, k, 2] < 0.5:  # Not visible
                continue
            hm = pred_heatmaps[b, k]
            idx = hm.argmax()
            pred_y = (idx // HEATMAP_SIZE).float() / HEATMAP_SIZE
            pred_x = (idx % HEATMAP_SIZE).float() / HEATMAP_SIZE

            gt_x = gt_keypoints[b, k, 0]
            gt_y = gt_keypoints[b, k, 1]

            dist = ((pred_x - gt_x)**2 + (pred_y - gt_y)**2).sqrt()
            if dist < threshold:
                correct += 1
            total += 1

    return correct / max(total, 1)


for epoch in range(start_epoch, EPOCHS):
    # ── Train ──
    model.train()
    train_loss = 0
    for images, heatmaps, kpts in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        images = images.to(device)
        heatmaps = heatmaps.to(device)

        pred = model(images)
        loss = criterion(pred, heatmaps)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    scheduler.step()

    # ── Validate ──
    model.eval()
    val_loss = 0
    val_pck = 0
    with torch.no_grad():
        for images, heatmaps, kpts in val_loader:
            images = images.to(device)
            heatmaps = heatmaps.to(device)

            pred = model(images)
            loss = criterion(pred, heatmaps)
            val_loss += loss.item()
            val_pck += pck_metric(pred.cpu(), kpts)

    val_loss /= len(val_loader)
    val_pck /= len(val_loader)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_pck'].append(val_pck)

    print(f'  Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | PCK@5%: {val_pck:.3f}')

    # Save best model locally
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f'{OUTPUT_DIR}/best_model.pth')
        # Also save best to Drive immediately
        shutil.copy(f'{OUTPUT_DIR}/best_model.pth', f'{DRIVE_CHECKPOINT_DIR}/best_model.pth')
        print(f'  ✓ Saved best model (val_loss={val_loss:.6f})')

    # Save checkpoint to Drive every N epochs
    if (epoch + 1) % CHECKPOINT_EVERY == 0:
        ckpt = {
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_loss': best_val_loss,
            'history': history,
        }
        torch.save(ckpt, f'{DRIVE_CHECKPOINT_DIR}/checkpoint_latest.pth')
        print(f'  💾 Checkpoint saved to Drive (epoch {epoch+1})')

print('\n✅ Training complete!')

In [ ]:
# ─── Training Curves ─────────────────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.legend()
ax1.set_title('Loss')

ax2.plot(history['val_pck'], color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('PCK@5%')
ax2.set_title('Keypoint Accuracy')
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150)
plt.show()

In [ ]:
# ─── Export to ONNX ──────────────────────────────────────
import onnx

# Load best model
model.load_state_dict(torch.load(f'{OUTPUT_DIR}/best_model.pth'))
model.eval()
model.cpu()

# Export
dummy = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE)
onnx_path = f'{OUTPUT_DIR}/spine_landmarks_cervical.onnx'

torch.onnx.export(
    model,
    dummy,
    onnx_path,
    opset_version=13,
    input_names=['image'],
    output_names=['heatmaps'],
    dynamic_axes={
        'image': {0: 'batch'},
        'heatmaps': {0: 'batch'},
    },
)

# Verify
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f'\n✓ ONNX model exported: {onnx_path}')
print(f'  Size: {size_mb:.1f} MB')
print(f'  Input: [1, 3, {INPUT_SIZE}, {INPUT_SIZE}]')
print(f'  Output: [1, {NUM_KEYPOINTS}, {HEATMAP_SIZE}, {HEATMAP_SIZE}]')

In [ ]:
# ─── Test ONNX Inference ─────────────────────────────────
import onnxruntime as ort

session = ort.InferenceSession(onnx_path)

# Run on a validation image
test_img, test_hm, test_kpts = val_ds[0]
pred = session.run(None, {'image': test_img.unsqueeze(0).numpy()})[0]

print(f'ONNX inference output shape: {pred.shape}')
print(f'Expected: (1, {NUM_KEYPOINTS}, {HEATMAP_SIZE}, {HEATMAP_SIZE})')

# Extract predicted coordinates
pred_coords = []
for k in range(NUM_KEYPOINTS):
    hm = pred[0, k]
    idx = hm.argmax()
    y = idx // HEATMAP_SIZE
    x = idx % HEATMAP_SIZE
    pred_coords.append((x / HEATMAP_SIZE, y / HEATMAP_SIZE))
    gt = test_kpts[k]
    if gt[2] > 0.5:
        dist = ((x/HEATMAP_SIZE - gt[0])**2 + (y/HEATMAP_SIZE - gt[1])**2)**0.5
        status = '✓' if dist < 0.05 else '✗'
        print(f'  {SPINEVIEW_LANDMARKS[k]:15s}: pred=({x/HEATMAP_SIZE:.3f}, {y/HEATMAP_SIZE:.3f}) gt=({gt[0]:.3f}, {gt[1]:.3f}) dist={dist:.4f} {status}')

In [ ]:
# ─── Visualise Predictions ───────────────────────────────
import matplotlib.pyplot as plt
import cv2

# Load original image
test_img_raw = cv2.imread(val_paths[0])
test_img_raw = cv2.cvtColor(test_img_raw, cv2.COLOR_BGR2RGB)
h, w = test_img_raw.shape[:2]

fig, ax = plt.subplots(1, 1, figsize=(8, 12))
ax.imshow(test_img_raw)

for k, (px, py) in enumerate(pred_coords):
    gt = test_kpts[k]
    if gt[2] > 0.5:
        # Ground truth (green)
        ax.plot(gt[0]*w, gt[1]*h, 'go', markersize=6, label='GT' if k == 0 else '')
        # Prediction (red)
        ax.plot(px*w, py*h, 'r+', markersize=8, label='Pred' if k == 0 else '')
        ax.annotate(SPINEVIEW_LANDMARKS[k].split('_')[0], (px*w, py*h),
                    fontsize=7, color='yellow', ha='left')

ax.legend()
ax.set_title('Landmark Predictions vs Ground Truth')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/prediction_sample.png', dpi=150)
plt.show()

In [ ]:
# ─── Save to Google Drive ────────────────────────────────
import shutil

drive_output = '/content/drive/MyDrive/SpineView_Model'
os.makedirs(drive_output, exist_ok=True)

shutil.copy(f'{OUTPUT_DIR}/spine_landmarks_cervical.onnx', drive_output)
shutil.copy(f'{OUTPUT_DIR}/best_model.pth', drive_output)
shutil.copy(f'{OUTPUT_DIR}/training_curves.png', drive_output)
shutil.copy(f'{OUTPUT_DIR}/prediction_sample.png', drive_output)

# Save landmark metadata for the browser runtime
metadata = {
    'model_name': 'spine_landmarks_cervical',
    'input_size': INPUT_SIZE,
    'heatmap_size': HEATMAP_SIZE,
    'num_keypoints': NUM_KEYPOINTS,
    'landmark_names': SPINEVIEW_LANDMARKS,
    'normalization': {
        'mean': [0.485, 0.456, 0.406],
        'std': [0.229, 0.224, 0.225],
    },
}
with open(f'{drive_output}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'\n✓ All files saved to Google Drive: {drive_output}')
print(f'\nFiles:')
for f in os.listdir(drive_output):
    size = os.path.getsize(f'{drive_output}/{f}') / (1024*1024)
    print(f'  {f} ({size:.1f} MB)')

print(f'\n🎉 Done! Download spine_landmarks_cervical.onnx and model_metadata.json')
print(f'   and place them in posture-pro/public/models/ for browser inference.')